# FSA/OWI — pobieranie obrazów par (do cache na Drive)

## Cel

Wczytuje `fsa_pairs.csv` z Drive, pobiera obrazy chosen+rejected przez IIIF (512×512), zapisuje lokalnie na Drive z czytelną konwencją nazw + manifest.

## Konwencja nazw

```
fsa_images/{loc_id}__{role}.jpg     # każdy obraz raz (dedup)
pairs_manifest.csv                  # pełne parowanie (pair_id -> chosen, rejected)
```

Rola (chosen/rejected) jest stabilna: chosen = titled, rejected = untitled, nigdy się nie mieszają. Jedna chosen może mieć wiele rejected (relacja 1:wiele) — manifest to spina, obraz zapisany raz.

## Cechy

- **IIIF 512** server-side, fallback: pobierz service_medium + resize lokalnie
- **Dedup** po loc_id — wspólne chosen pobierane raz
- **Cache** — pomija już pobrane pliki (wznawialne)
- **Manifest przyrostowy** — zapis co N obrazów
- **Rate limit** — łagodny dla image CDN


## 0. Drive + konfiguracja

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import time, re, csv
from pathlib import Path
from io import BytesIO
import requests
import pandas as pd
from PIL import Image

DRIVE_DIR = Path('/content/drive/MyDrive/fsa_data')
PAIRS_CSV = DRIVE_DIR / 'fsa_pairs.csv'
IMAGES_DIR = DRIVE_DIR / 'fsa_images'
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST = DRIVE_DIR / 'pairs_manifest.csv'

SIZE = 512                 # docelowy rozmiar (fit within SIZExSIZE)
IMG_DELAY = 0.3            # przerwa między pobraniami (image CDN tolerancyjny)
MAX_PAIRS = None           # None = wszystkie; lub liczba do testu (np. 500)
SAVE_EVERY = 50            # zapis manifestu co N przetworzonych par

HEADERS = {'User-Agent': 'DecisiveMoment-Research/1.0 (academic)'}

print(f'Pairs:   {PAIRS_CSV}')
print(f'Images:  {IMAGES_DIR}')
print(f'Rozmiar: {SIZE}, delay: {IMG_DELAY}s')


## 1. Wczytaj pary, wyciągnij loc_id

In [ ]:
pairs = pd.read_csv(PAIRS_CSV, dtype=str, keep_default_na=False)
if MAX_PAIRS:
    pairs = pairs.head(MAX_PAIRS)
print(f'Par: {len(pairs)}')

def loc_id(id_url):
    """http://www.loc.gov/item/2024665975/ -> 2024665975"""
    m = re.search(r'/item/([^/]+)', str(id_url))
    return m.group(1) if m else re.sub(r'[^A-Za-z0-9]', '', str(id_url))[:20]

pairs['chosen_locid'] = pairs['chosen_id'].apply(loc_id)
pairs['rejected_locid'] = pairs['rejected_id'].apply(loc_id)

# Unikalne obrazy do pobrania (dedup po loc_id) z rolą i URL
to_download = {}  # loc_id -> (role, url)
for _, r in pairs.iterrows():
    to_download[r['chosen_locid']] = ('chosen', r['chosen_img'])
    to_download[r['rejected_locid']] = ('rejected', r['rejected_img'])

print(f'Unikalnych obrazów do pobrania: {len(to_download)}')
print(f'  (chosen wspólne dla wielu par pobierane raz)')


## 2. Funkcja pobierania — IIIF + fallback resize

In [ ]:
def service_to_iiif(url, size=SIZE):
    """Konstruuje IIIF URL (server-side resize) z service_medium URL.
    
    .../storage-services/service/pnp/cph/.../3a19325r.jpg
    -> .../image-services/iiif/service:pnp:cph:...:3a19325/full/!512,512/0/default.jpg
    """
    m = re.search(r'storage-services/(.+)\.jpg$', str(url))
    if not m:
        return None
    path = m.group(1)              # service/pnp/cph/.../3a19325r
    path = re.sub(r'r$', '', path)  # usuń sufiks rendition 'r'
    ident = path.replace('/', ':')
    return f'https://tile.loc.gov/image-services/iiif/{ident}/full/!{size},{size}/0/default.jpg'

def save_resized(content, path, size=SIZE):
    """Zapisz obraz przeskalowany do fit-within size×size."""
    img = Image.open(BytesIO(content)).convert('RGB')
    img.thumbnail((size, size), Image.LANCZOS)
    img.save(path, 'JPEG', quality=90)

def download_image(url, out_path, size=SIZE):
    """Pobiera obraz: IIIF (server-side) najpierw, fallback service_medium+resize.
    Zwraca 'iiif' / 'fallback' / None (porażka).
    """
    # 1. Próba IIIF (server-side 512, oszczędza pasmo)
    iiif = service_to_iiif(url, size)
    if iiif:
        try:
            r = requests.get(iiif, headers=HEADERS, timeout=30)
            if r.status_code == 200 and 'image' in r.headers.get('content-type', ''):
                with open(out_path, 'wb') as f:
                    f.write(r.content)
                return 'iiif'
        except Exception:
            pass
    # 2. Fallback: pobierz oryginalny URL i resize lokalnie
    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        if r.status_code == 200 and 'image' in r.headers.get('content-type', ''):
            save_resized(r.content, out_path, size)
            return 'fallback'
    except Exception as e:
        print(f'    ⚠ {e}')
    return None

# Szybki test na jednym obrazie
test_locid, (test_role, test_url) = next(iter(to_download.items()))
test_path = IMAGES_DIR / f'_test_{test_locid}.jpg'
print(f'Test pobierania: {test_locid} ({test_role})')
method = download_image(test_url, test_path)
if method:
    img = Image.open(test_path)
    print(f'  ✅ Pobrano metodą "{method}", rozmiar {img.size}')
    test_path.unlink()  # usuń testowy
else:
    print(f'  ✗ Nie udało się — sprawdź URL: {test_url[:80]}')


## 3. Główne pobieranie (z cache i wznawianiem)

In [ ]:
from tqdm.auto import tqdm

stats = {'iiif': 0, 'fallback': 0, 'cached': 0, 'failed': 0}
items = list(to_download.items())

for i, (lid, (role, url)) in enumerate(tqdm(items, desc='Pobieranie')):
    out_path = IMAGES_DIR / f'{lid}__{role}.jpg'
    if out_path.exists() and out_path.stat().st_size > 0:
        stats['cached'] += 1
        continue
    if not url:
        stats['failed'] += 1
        continue
    time.sleep(IMG_DELAY)
    method = download_image(url, out_path)
    if method:
        stats[method] += 1
    else:
        stats['failed'] += 1

print(f'\n✅ Pobieranie zakończone:')
for k, v in stats.items():
    print(f'  {k}: {v}')
print(f'  Plików w {IMAGES_DIR.name}: {len(list(IMAGES_DIR.glob("*.jpg")))}')


## 4. Zapisz manifest par

In [ ]:
manifest_rows = []
for idx, r in pairs.reset_index(drop=True).iterrows():
    c_file = f'{r["chosen_locid"]}__chosen.jpg'
    rej_file = f'{r["rejected_locid"]}__rejected.jpg'
    c_ok = (IMAGES_DIR / c_file).exists()
    rej_ok = (IMAGES_DIR / rej_file).exists()
    manifest_rows.append({
        'pair_id': f'{idx:06d}',
        'chosen_locid': r['chosen_locid'],
        'rejected_locid': r['rejected_locid'],
        'chosen_file': c_file,
        'rejected_file': rej_file,
        'both_ok': c_ok and rej_ok,
        'scene': r.get('scene', ''),
        'method': r.get('method', ''),
    })

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(MANIFEST, index=False)

ok = manifest['both_ok'].sum()
print(f'✅ Manifest: {MANIFEST}')
print(f'  Par łącznie:        {len(manifest)}')
print(f'  Par z obu obrazami: {ok} ({ok*100//max(len(manifest),1)}%)')
print(f'  Par niekompletnych: {len(manifest)-ok}')
print(f'\nKolumny manifestu: {list(manifest.columns)}')
print('\nUżycie: dla pary wczytaj IMAGES_DIR/chosen_file i IMAGES_DIR/rejected_file')


## 5. Podgląd kilku pobranych par

In [ ]:
import matplotlib.pyplot as plt

ok_pairs = manifest[manifest['both_ok']].head(4).reset_index(drop=True)
N = len(ok_pairs)
if N > 0:
    fig, axes = plt.subplots(N, 2, figsize=(8, 4 * N))
    if N == 1: axes = axes.reshape(1, 2)
    for i in range(N):
        row = ok_pairs.iloc[i]
        ic = Image.open(IMAGES_DIR / row['chosen_file'])
        ir = Image.open(IMAGES_DIR / row['rejected_file'])
        axes[i,0].imshow(ic); axes[i,0].set_title(f'CHOSEN\n{row["chosen_file"]}', fontsize=8, color='green'); axes[i,0].axis('off')
        axes[i,1].imshow(ir); axes[i,1].set_title(f'REJECTED\n{row["rejected_file"]}', fontsize=8, color='red'); axes[i,1].axis('off')
    plt.suptitle('Pobrane pary (512px)', fontsize=12, fontweight='bold', y=1.003)
    plt.tight_layout()
    plt.show()
else:
    print('Brak kompletnych par do podglądu.')


## 6. Co dalej

Masz teraz lokalnie na Drive:
- `fsa_images/{loc_id}__{role}.jpg` — obrazy 512px
- `pairs_manifest.csv` — parowanie

Wszystkie kolejne kroki czytają z dysku (zero API):
1. **Filtr CLIP similarity** — embeddingi obrazów, cosine chosen vs rejected, odrzuć różne sceny
2. **Akt 1** — decisive_direction, d', linear probe
3. **Akt 2** — interpretacja osi (CLIP-text)

### Wznawianie
Jeśli pobieranie przerwane: uruchom ponownie komórkę 3 — cache pomija pobrane pliki, dociąga brakujące.

### Test na małej próbce
Ustaw `MAX_PAIRS = 500` w konfiguracji, by przetestować pipeline zanim pobierzesz wszystko.
